# Computing the collision probabilities of different unique DNA molecules into the same consensus molecule via equal cuts and equal tagging

### How to compute required input

#### (optional) 0. Sort the files in case they are not sorted
This might not be needed for the immediate next step but might be useful for exploring some example cases

```
# in case you want to resort the BAM file
samtools sort -@ 7 P_6_2_J_1.sorted.bam -o P_6_2_J_1.resorted.bam --write-index
```

#### 1. Starting from a BAM file you can run this 
This will report for each read which is the starting and end position of the alignment.

```
for file in $(ls *.sorted.bam); do
  sample=${file%.bam}
  echo "$sample"

  samtools view -@ 15 ${sample}.bam | \
  awk '
  function cigar_len(cigar,   len,op,total) {
      total=0
      while (match(cigar, /[0-9]+[MIDNSHP=X]/)) {
          len=substr(cigar, RSTART, RLENGTH-1)
          op=substr(cigar, RSTART+RLENGTH-1, 1)
          if (op ~ /[MDN=X]/) total += len
          cigar=substr(cigar, RSTART+RLENGTH)
      }
      return total
  }

  BEGIN{OFS="\t"}

  {
    chr=$3
    pos=$4
    cigar=$6

    if (chr=="*" || cigar=="*") next

    len = cigar_len(cigar)
    end = pos + len - 1

    print $1, chr, pos, end
  }' | gzip > ${sample}.read_coords.tsv.gz

done
```


#### 2. Merge the information of each pair of reads and report how many times a given set of cuts is unique
```
for file in $(ls *_2.sorted.bam | cut -d '.' -f1); do
echo $file;
python /data/bbg/projects/prominent/protocols/paper_figures/supplementary_analysis/UMIcollisions/compute_unique_cuts_freq.py $file /data/bbg/nobackup2/prominent/duplex_seq_tests/error_rate/cord_blood/bbg/2026-03-17_tws_idt ;
done;
```

Example of output:

```
$ zcat SC001_B1_1_H_2.sorted.read_coords.cuts_freq.tsv.gz
repeats	frequency
1	46,353,191
2	2,430,718
3	254,679
4	37,477
5	7,193
6	1,687
7	485
8	191
9	80
10	39
11	19
12	9
14	6
16	3
13	2
18	1
17	1
```


In [1]:
import numpy as np
import pandas as pd
from scipy.optimize import brentq

import matplotlib.pyplot as plt

import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
run = "/data/bbg/nobackup2/prominent/duplex_seq_tests/error_rate/cord_blood/bbg/2026-03-17_tws_idt"

In [ ]:
cord_blood_samples_list = ["SC003_B1_1_H_1", "SC002_B1_1_H_1", "SC001_B1_1_H_1",
                           "SC003_B1_1_H_2", "SC002_B1_1_H_2", "SC001_B1_1_H_2"]
idt_cord_blood_samples_list = ["SC003_B1_1_H_2", "SC002_B1_1_H_2", "SC001_B1_1_H_2"]
reference_sample = "SC001_B1_1_H_2"


## Read cut frequency files and plot them

In [4]:
all_counts_df = pd.DataFrame()
for file in os.listdir(f"{run}/processing_files/sortbamamfiltered/"):
    if file.endswith(".sorted.read_coords.cuts_freq.tsv.gz"):
        print(file)
        sample = file.split('.')[0]
        sample_counts = pd.read_csv(f"{run}/processing_files/sortbamamfiltered/{file}", sep="\t")
        sample_counts["sample"] = sample
        sample_counts["relative_frequency"] = sample_counts["frequency"] / sample_counts["frequency"].sum()
        # sample_counts["expected_poisson_cuts"] = (sample_counts["relative_frequency"] * sample_counts["repeats"]).sum()
        all_counts_df = pd.concat([all_counts_df, sample_counts], ignore_index=True)


SC002_B1_1_H_2.sorted.read_coords.cuts_freq.tsv.gz
SC001_B1_1_H_2.sorted.read_coords.cuts_freq.tsv.gz
SC003_B1_1_H_2.sorted.read_coords.cuts_freq.tsv.gz
SC001_B1_1_H_1.sorted.read_coords.cuts_freq.tsv.gz
SC002_B1_1_H_1.sorted.read_coords.cuts_freq.tsv.gz
SC003_B1_1_H_1.sorted.read_coords.cuts_freq.tsv.gz


In [5]:
all_counts_df["total_fragments"] = all_counts_df["repeats"] * all_counts_df["frequency"]
all_counts_df["repeated"] = all_counts_df["repeats"] > 1
repeated_frequency = all_counts_df.groupby(by = ["sample", "repeated"])["total_fragments"].sum()
repeated_frequency

sample          repeated
SC001_B1_1_H_1  False       72605095
                True         9930665
SC001_B1_1_H_2  False       46353191
                True         5828011
SC002_B1_1_H_1  False       91013953
                True        13236728
SC002_B1_1_H_2  False       57106533
                True         8601399
SC003_B1_1_H_1  False       97322864
                True        14360079
SC003_B1_1_H_2  False       68727524
                True        15478474
Name: total_fragments, dtype: int64

In [6]:
all_counts_df.head(20)

,repeats,frequency,sample,relative_frequency,total_fragments,repeated
0,1,57106533,SC002_B1_1_H_2,9.348416e-01,57106533,False
1,2,3463947,SC002_B1_1_H_2,5.670528e-02,6927894,True
2,3,424075,SC002_B1_1_H_2,6.942164e-03,1272225,True
3,4,70498,SC002_B1_1_H_2,1.154062e-03,281992,True
4,5,15412,SC002_B1_1_H_2,2.522965e-04,77060,True
5,6,4239,SC002_B1_1_H_2,6.939300e-05,25434,True
6,7,1288,SC002_B1_1_H_2,2.108473e-05,9016,True
7,8,502,SC002_B1_1_H_2,8.217807e-06,4016,True
8,9,176,SC002_B1_1_H_2,2.881144e-06,1584,True
9,10,76,SC002_B1_1_H_2,1.244130e-06,760,True


## UMI counts

In [7]:
def get_single_umi_probabilities_vector(sample, run, plot = False, verbose=False):
    umi_counts_data = pd.read_table(f"{run}/metrics/duplex/fgbio_seqmetricsontarget/{sample}_chr1.on_target.duplex_seq_metrics.umi_counts.txt")
    umi_counts_data = umi_counts_data.sort_values("fraction_unique_observations", ascending = False).reset_index(drop=True)
    umi_counts_data["cummulative_fraction"] = umi_counts_data["fraction_unique_observations"].cumsum()

    # identify the elbow point in the cumulative distribution of UMIs to determine a cutoff for high-frequency UMIs
    init_val = 1
    i = 0
    diff = 0
    while diff < (init_val / 2):
        new_val = umi_counts_data.loc[i,"fraction_unique_observations"]
        if verbose:
            print(i, new_val, init_val)
        if i == 0:
            init_val = umi_counts_data.loc[i,"fraction_unique_observations"]
        else:
            diff = init_val - umi_counts_data.loc[i,"fraction_unique_observations"]
        init_val = umi_counts_data.loc[i,"fraction_unique_observations"]
        if verbose:
            print(diff)
        i += 1

    umi_tag_probabilities = umi_counts_data.loc[:i-1,"fraction_unique_observations"].values
    umi_tag_probabilities_norm = umi_tag_probabilities / umi_tag_probabilities.sum()

    return i, umi_tag_probabilities, umi_tag_probabilities_norm


In [8]:
def get_duplex_umi_probabilities_vector(sample, run, size, plot = False, verbose=False):
    umi_counts_data = pd.read_table(f"{run}/metrics/duplex/fgbio_seqmetricsontarget/{sample}_chr1.on_target.duplex_seq_metrics.duplex_umi_counts.txt")
    umi_counts_data = umi_counts_data.sort_values("fraction_unique_observations", ascending = False).reset_index(drop=True)
    umi_counts_data["cummulative_fraction"] = umi_counts_data["fraction_unique_observations"].cumsum()

    umi_tag_probabilities = umi_counts_data.loc[:size,"fraction_unique_observations"].values
    umi_tag_probabilities_norm = umi_tag_probabilities / umi_tag_probabilities.sum()

    return umi_tag_probabilities, umi_tag_probabilities_norm


## Estimate UMI collisions

In [9]:
def posterior_expectation_one_repeat(p):
    """
    p: numpy array of probabilities
    """
    num = np.sum(p / ((1 - p)**2))
    den = np.sum(p / (1 - p))
    return num / den

# expected_N = posterior_expectation_one_repeat(p)
# expected_N

In [10]:
def func(N, p):
    """
    p: numpy array of probabilities
    N: number of independent trials
    """
    return np.sum(1 - (1 - p) ** N)


def find_N(p, target):
    """
    p: numpy array of probabilities
    target: target value for the sum
    """
    # Define a function that we want to find the root of
    def objective(N):
        return func(N, p) - target
    
    # Use brentq to find the root of the objective function
    N_solution = brentq(objective, 1, 1e6)  # Search for N in the range [1, 1e6]
    
    return N_solution

In [11]:
def compute_original_fragments_per_sample(counts_df, vector_of_probabilities):
    n_repeats_observed = counts_df["repeats"].unique().tolist()
    n_repeats_observed.remove(1)

    mapping_observed_repeats_to_original_cuts = {1: posterior_expectation_one_repeat(vector_of_probabilities).item()}
    
    for n_repeats in n_repeats_observed:
        N_solution = find_N(vector_of_probabilities, n_repeats)
        mapping_observed_repeats_to_original_cuts[n_repeats] = N_solution

    print(mapping_observed_repeats_to_original_cuts)
    return mapping_observed_repeats_to_original_cuts

In [12]:
def plot_umi_frequencies(x, type_of_umi = "duplex", sample = "sample"):
    data = pd.DataFrame(sorted(x, reverse=True))
    data.columns = ["frequency"]
    data = data.reset_index()

    # plot cumsumative distribution of duplex UMIs (fraction_raw_observations)
    sns.lineplot(data=data, x="index", y="frequency")
    plt.xlabel(f"{type_of_umi.capitalize()} UMI Rank")
    plt.ylabel("Fraction of Raw Observations")
    plt.title(f"{sample}\nCumulative Distribution of {type_of_umi.capitalize()} UMIs")
    # plt.xlim(-100, 15000)
    plt.show()

In [28]:
umi_structure_16_16 = np.array([1/16**2] * 16**2)
umi_structure_32_32 = np.array([1/32**2] * 32**2)
umi_structure_64_64 = np.array([1/64**2] * 64**2)
umi_structure_96_96 = np.array([1/96**2] * 96**2)

In [41]:
upd_counts_df = pd.DataFrame()
# compile all metrics in a single table
summary_stats = []
for sample in idt_cord_blood_samples_list:

    unique_tags_number, umi_tag_probabilities, umi_tag_probabilities_norm = get_single_umi_probabilities_vector(sample, run, plot = False, verbose=False)

    print(f"{sample}: main_UMI_tags\t= {unique_tags_number}")
    tag_probabilities_assume_random_side = np.matmul(umi_tag_probabilities_norm.reshape(-1, 1), umi_tag_probabilities_norm.reshape(1, -1)).flatten()

    duplex_tag_probabilities, duplex_tag_probabilities_norm = get_duplex_umi_probabilities_vector(sample, run,
                                                                                                  size=unique_tags_number**2,
                                                                                                  verbose=False)

    vector_of_probabilities = tag_probabilities_assume_random_side
    # vector_of_probabilities = np.array([1/unique_tags_number**2] * unique_tags_number**2)

    # plot_umi_frequencies(umi_tag_probabilities_norm, type_of_umi="single", sample=sample)
    # plot_umi_frequencies(vector_of_probabilities, type_of_umi="duplex", sample=sample)
    # plot_umi_frequencies(duplex_tag_probabilities_norm, type_of_umi="duplex observed", sample=sample)

    for unique_tags, prob_vector in [(unique_tags_number, vector_of_probabilities),
                         (16, umi_structure_16_16),
                         (32, umi_structure_32_32),
                         (64, umi_structure_64_64),
                         (96, umi_structure_96_96)]:
        probability_of_same_tag_2_frag = np.sum(prob_vector ** 2)
        D_eff = 1 / probability_of_same_tag_2_frag
        print(f"{sample}: D_eff\t\t= {D_eff:.0f}")


        sample_count_df = all_counts_df[all_counts_df["sample"] == sample].copy()
    
        mapping_observed_repeats_to_original_cuts = compute_original_fragments_per_sample(sample_count_df, prob_vector)

        sample_count_df["original_fragments_per_cutsite"] = sample_count_df["repeats"].map(mapping_observed_repeats_to_original_cuts)
        break
    break

        # sample_count_df["original_fragments"] = sample_count_df["original_fragments_per_cutsite"] * sample_count_df["frequency"]
        # lost_fragments = sample_count_df["original_fragments"].sum() - sample_count_df["total_fragments"].sum()

        # print(f"{sample}: Lost fragments\t= {lost_fragments:,.0f}")

        # lost_proportion = lost_fragments / sample_count_df["original_fragments"].sum()

        # print(f"{sample}: Lost proportion\t= {lost_proportion:.2%}")
        # upd_counts_df = pd.concat([upd_counts_df, sample_count_df], ignore_index=True)
        # summary_stats.append([sample, unique_tags, unique_tags**2, round(D_eff), round(lost_fragments), lost_proportion])

        # print()


SC003_B1_1_H_2: main_UMI_tags	= 32
SC003_B1_1_H_2: D_eff		= 443
{1: 1.0022711630133867, 2: 2.0022657230900776, 3: 3.006803439694831, 4: 4.013619438468004, 5: 5.022720026359143, 6: 6.034111528687481, 7: 7.04780028921133, 8: 8.063792670200078, 9: 9.082095052506372, 10: 10.10271383563894, 11: 11.125655437834645, 12: 12.150926296135601, 13: 13.178532866459909, 14: 14.208481623678706, 15: 15.240779061691283, 16: 16.27543169350133, 17: 17.312446051293684, 18: 18.35182868651172, 20: 20.437725091759482, 23: 23.584496682479436, 22: 22.53317370894082, 19: 19.393586169935322, 24: 24.63822765094298, 21: 21.484252061673516, 39: 40.737747612182986, 74: 80.54840182729684, 25: 25.69437330280249, 33: 34.23127634200087, 26: 26.752940346428655}


In [35]:
summary_stats_df = pd.DataFrame(summary_stats, columns=["sample", "unique_tags_number", "unique_tags_number_duplex", "D_eff",
                                                        "lost_fragments", "lost_proportion"])
summary_stats_df.sort_values(by=["sample", "unique_tags_number"])

,sample,unique_tags_number,unique_tags_number_duplex,D_eff,lost_fragments,lost_proportion


In [42]:
sample_count_df

,repeats,frequency,sample,relative_frequency,total_fragments,repeated,original_fragments_per_cutsite
38,1,68727524,SC003_B1_1_H_2,9.091872e-01,68727524,False,1.002271
39,2,5599366,SC003_B1_1_H_2,7.407326e-02,11198732,True,2.002266
40,3,947734,SC003_B1_1_H_2,1.253745e-02,2843202,True,3.006803
41,4,218450,SC003_B1_1_H_2,2.889846e-03,873800,True,4.013619
42,5,62566,SC003_B1_1_H_2,8.276773e-04,312830,True,5.022720
43,6,21282,SC003_B1_1_H_2,2.815367e-04,127692,True,6.034112
44,7,8382,SC003_B1_1_H_2,1.108844e-04,58674,True,7.047800
45,8,3588,SC003_B1_1_H_2,4.746517e-05,28704,True,8.063793
46,9,1599,SC003_B1_1_H_2,2.115296e-05,14391,True,9.082095
47,10,783,SC003_B1_1_H_2,1.035820e-05,7830,True,10.102714


In [45]:
sample_count_df[["repeats", "frequency"]].values

array([[       1, 68727524],
       [       2,  5599366],
       [       3,   947734],
       [       4,   218450],
       [       5,    62566],
       [       6,    21282],
       [       7,     8382],
       [       8,     3588],
       [       9,     1599],
       [      10,      783],
       [      11,      397],
       [      12,      222],
       [      13,      163],
       [      14,       81],
       [      15,       45],
       [      16,       22],
       [      17,       19],
       [      18,       13],
       [      20,        7],
       [      23,        5],
       [      22,        4],
       [      19,        4],
       [      24,        3],
       [      21,        3],
       [      39,        1],
       [      74,        1],
       [      25,        1],
       [      33,        1],
       [      26,        1]])